# Chapter 5.3: Temporal and Dynamic Models for Sequential Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement **TiSASRec** (Time-Interval Aware Self-Attention) to incorporate time gaps between interactions
2. Compare **absolute positional encoding** vs **relative time-gap encoding**
3. Build dynamic user representations that evolve over time
4. Understand **temporal point processes** for modeling event timing
5. Capture seasonal and periodic patterns in user behavior
6. Design time-aware attention mechanisms that weight recent interactions more heavily
7. Evaluate the impact of temporal information on recommendation quality

## Prerequisites

- Chapter 5.2 (SASRec architecture)
- Understanding of self-attention and positional encoding
- Basic probability (exponential distributions, intensity functions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part5/chapter_5.3_temporal_models.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part5/chapter_5.3_temporal_models.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import math
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Why Time Matters in Sequential Recommendation

Standard SASRec/BERT4Rec use **positional embeddings** that encode the *order* of items but ignore the *actual time gaps*. Consider:

- User clicks items A, B within 2 minutes (strong intent)
- User clicks item C 3 days later (likely different intent)

SASRec treats positions 1,2,3 the same regardless of time gaps. **TiSASRec** fixes this.

> **💡 Concept:** The time interval between two interactions encodes important signal about user intent continuity. Short gaps suggest related browsing; long gaps suggest context switches.

## 2. Synthetic Temporal Data

In [ ]:
def generate_temporal_sequences(n_users=800, n_items=500, min_len=5, max_len=40, seed=42):
    """Generate user sequences with timestamps (varying time gaps)."""
    rng = np.random.RandomState(seed)
    n_categories = 20
    item_category = rng.randint(0, n_categories, size=n_items)
    item_popularity = rng.power(0.5, size=n_items)
    item_popularity /= item_popularity.sum()
    
    data = []
    for u in range(n_users):
        seq_len = rng.randint(min_len, max_len + 1)
        user_cats = rng.choice(n_categories, size=3, replace=False)
        
        items = []
        timestamps = []
        t = rng.uniform(0, 100)  # Starting time (days)
        
        for step in range(seq_len):
            # Time gap: mix of short (minutes) and long (days) gaps
            if rng.random() < 0.6:
                gap = rng.exponential(0.01)  # Short gap (~minutes in days)
            else:
                gap = rng.exponential(2.0)  # Long gap (~days)
            t += gap
            
            # Item selection with category coherence for short gaps
            if gap < 0.05 and len(items) > 0:  # Short gap -> same category
                last_cat = item_category[items[-1]]
                cat_items = np.where(item_category == last_cat)[0]
                if len(cat_items) > 0:
                    probs = item_popularity[cat_items]
                    probs /= probs.sum()
                    item = rng.choice(cat_items, p=probs)
                else:
                    item = rng.choice(n_items, p=item_popularity)
            else:
                cat = rng.choice(user_cats)
                cat_items = np.where(item_category == cat)[0]
                probs = item_popularity[cat_items]
                probs /= probs.sum()
                item = rng.choice(cat_items, p=probs)
            
            items.append(int(item))
            timestamps.append(float(t))
        
        data.append({"items": items, "timestamps": timestamps})
    
    return data

N_ITEMS = 500
MAX_SEQ_LEN = 40
temporal_data = generate_temporal_sequences(n_users=800, n_items=N_ITEMS)

# Compute time intervals
all_intervals = []
for d in temporal_data:
    ts = d["timestamps"]
    intervals = [ts[i+1] - ts[i] for i in range(len(ts)-1)]
    all_intervals.extend(intervals)

print(f"Users: {len(temporal_data)}, Items: {N_ITEMS}")
print(f"Time intervals — mean: {np.mean(all_intervals):.3f} days, "
      f"median: {np.median(all_intervals):.3f} days")

In [ ]:
# Visualize time interval distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(all_intervals, bins=100, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Time Interval (days)")
axes[0].set_ylabel("Count")
axes[0].set_title("Time Interval Distribution")
axes[0].set_xlim(0, 10)

# Log scale
log_intervals = np.log1p(all_intervals)
axes[1].hist(log_intervals, bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("log(1 + interval)")
axes[1].set_ylabel("Count")
axes[1].set_title("Log Time Interval Distribution")

plt.tight_layout()
plt.show()

## 3. Time Interval Encoding

Two approaches to encoding time information:

### Approach 1: Absolute Time Encoding
Encode the absolute timestamp using sinusoidal functions (like positional encoding):

$$\text{TE}(t, 2i) = \sin(t / 10000^{2i/d})$$
$$\text{TE}(t, 2i+1) = \cos(t / 10000^{2i/d})$$

### Approach 2: Relative Time-Gap Encoding (TiSASRec)
Encode the time gap between the current and each previous interaction:

$$\Delta t_{ij} = |t_i - t_j|$$

Bucket the time gaps into discrete intervals and learn embeddings per bucket.

In [ ]:
class AbsoluteTimeEncoding(nn.Module):
    """Sinusoidal encoding for absolute timestamps."""
    
    def __init__(self, embed_dim, max_period=10000.0):
        super().__init__()
        self.embed_dim = embed_dim
        # Pre-compute frequency bands
        half_dim = embed_dim // 2
        freqs = torch.exp(
            -torch.arange(half_dim, dtype=torch.float) * math.log(max_period) / half_dim
        )
        self.register_buffer("freqs", freqs)
    
    def forward(self, timestamps):
        """
        timestamps: (batch, seq_len) float tensor
        Returns: (batch, seq_len, embed_dim)
        """
        angles = timestamps.unsqueeze(-1) * self.freqs  # (batch, seq_len, half_dim)
        encoding = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
        return encoding

class RelativeTimeGapEncoding(nn.Module):
    """Learnable embeddings for bucketed time intervals (TiSASRec-style)."""
    
    def __init__(self, embed_dim, n_buckets=64):
        super().__init__()
        self.n_buckets = n_buckets
        self.time_emb = nn.Embedding(n_buckets, embed_dim)
        nn.init.xavier_uniform_(self.time_emb.weight)
    
    def _bucket_time_gaps(self, time_gaps):
        """Map continuous time gaps to discrete buckets using log-scale."""
        # Log-scale bucketing: bucket 0 = gap < 1min, bucket 63 = gap > 1year
        log_gaps = torch.log1p(time_gaps * 1440)  # Convert days to minutes, then log
        max_log = math.log1p(365 * 1440)  # Max: 1 year in minutes
        bucket_ids = (log_gaps / max_log * (self.n_buckets - 1)).long()
        bucket_ids = bucket_ids.clamp(0, self.n_buckets - 1)
        return bucket_ids
    
    def forward(self, time_gaps):
        """
        time_gaps: (batch, seq_len, seq_len) pairwise time gaps
        Returns: (batch, seq_len, seq_len, embed_dim) time-gap embeddings
        """
        bucket_ids = self._bucket_time_gaps(time_gaps)
        return self.time_emb(bucket_ids)

# Demo
abs_enc = AbsoluteTimeEncoding(embed_dim=64)
demo_times = torch.tensor([[0.0, 0.01, 0.5, 3.0, 3.02]])
abs_out = abs_enc(demo_times)
print(f"Absolute time encoding shape: {abs_out.shape}")

rel_enc = RelativeTimeGapEncoding(embed_dim=64, n_buckets=64)
# Pairwise time gaps
gaps = torch.abs(demo_times.unsqueeze(-1) - demo_times.unsqueeze(-2))  # (1, 5, 5)
rel_out = rel_enc(gaps)
print(f"Relative time-gap encoding shape: {rel_out.shape}")

## 4. TiSASRec: Time-Interval Aware Self-Attention

**TiSASRec** (Li et al., WSDM 2020) modifies the attention computation to incorporate relative time intervals:

$$\alpha_{ij} = \frac{(\mathbf{q}_i \cdot \mathbf{k}_j + \mathbf{q}_i \cdot \mathbf{r}^K_{|i-j|} + \mathbf{q}_i \cdot \mathbf{t}^K_{\Delta t_{ij}})}{\sqrt{d}}$$

$$\mathbf{o}_i = \sum_j \text{softmax}(\alpha_{ij})(\mathbf{v}_j + \mathbf{r}^V_{|i-j|} + \mathbf{t}^V_{\Delta t_{ij}})$$

where:
- $\mathbf{r}^K_{|i-j|}, \mathbf{r}^V_{|i-j|}$: relative position embeddings
- $\mathbf{t}^K_{\Delta t_{ij}}, \mathbf{t}^V_{\Delta t_{ij}}$: time interval embeddings

> **💡 Concept:** TiSASRec uses **two types** of relative encodings: positional (order-based) and temporal (time-based). Both are added to the key and value in attention, not to the input embeddings.

In [ ]:
class TimeAwareAttention(nn.Module):
    """Time-interval aware multi-head self-attention (TiSASRec-style)."""
    
    def __init__(self, embed_dim, n_heads, max_len=40, n_time_buckets=64, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        assert embed_dim % n_heads == 0
        
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        
        # Relative position embeddings
        self.pos_k = nn.Embedding(2 * max_len - 1, self.head_dim)
        self.pos_v = nn.Embedding(2 * max_len - 1, self.head_dim)
        
        # Time interval embeddings
        self.time_k = nn.Embedding(n_time_buckets, self.head_dim)
        self.time_v = nn.Embedding(n_time_buckets, self.head_dim)
        
        self.n_time_buckets = n_time_buckets
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)
    
    def _bucket_time(self, time_gaps):
        """Map time gaps to bucket IDs."""
        log_gaps = torch.log1p(time_gaps * 1440)
        max_log = math.log1p(365 * 1440)
        bucket_ids = (log_gaps / max_log * (self.n_time_buckets - 1)).long()
        return bucket_ids.clamp(0, self.n_time_buckets - 1)
    
    def forward(self, x, time_gaps, causal_mask=None):
        """
        x: (batch, seq_len, embed_dim)
        time_gaps: (batch, seq_len, seq_len) pairwise absolute time differences
        causal_mask: (seq_len, seq_len) boolean mask
        """
        B, L, D = x.shape
        H = self.n_heads
        d_k = self.head_dim
        
        Q = self.W_q(x).view(B, L, H, d_k).transpose(1, 2)  # (B, H, L, d_k)
        K = self.W_k(x).view(B, L, H, d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, H, d_k).transpose(1, 2)
        
        # Content-based attention
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B, H, L, L)
        
        # Relative position bias
        positions = torch.arange(L, device=x.device)
        rel_pos = positions.unsqueeze(0) - positions.unsqueeze(1) + self.max_len - 1  # (L, L)
        rel_pos = rel_pos.clamp(0, 2 * self.max_len - 2)
        rk = self.pos_k(rel_pos)  # (L, L, d_k)
        pos_bias = torch.einsum("bhqd,qkd->bhqk", Q, rk) / self.scale
        attn_scores = attn_scores + pos_bias
        
        # Time interval bias
        time_bucket_ids = self._bucket_time(time_gaps)  # (B, L, L)
        tk = self.time_k(time_bucket_ids)  # (B, L, L, d_k)
        time_bias = torch.einsum("bhqd,bqkd->bhqk", Q, tk) / self.scale
        attn_scores = attn_scores + time_bias
        
        # Apply causal mask
        if causal_mask is not None:
            attn_scores = attn_scores.masked_fill(
                causal_mask.unsqueeze(0).unsqueeze(0), float("-inf")
            )
        
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum of values (with position and time value biases)
        rv = self.pos_v(rel_pos)  # (L, L, d_k)
        tv = self.time_v(time_bucket_ids)  # (B, L, L, d_k)
        
        context = torch.matmul(attn_weights, V)  # (B, H, L, d_k)
        # Add position value bias
        context = context + torch.einsum("bhqk,qkd->bhqd", attn_weights, rv)
        # Add time value bias
        context = context + torch.einsum("bhqk,bqkd->bhqd", attn_weights, tv)
        
        context = context.transpose(1, 2).contiguous().view(B, L, D)
        return self.W_o(context)

# Test
time_attn = TimeAwareAttention(embed_dim=64, n_heads=2, max_len=MAX_SEQ_LEN)
test_x = torch.randn(2, 10, 64)
test_gaps = torch.rand(2, 10, 10) * 5  # Random time gaps in days
out = time_attn(test_x, test_gaps)
print(f"TimeAwareAttention output shape: {out.shape}")

In [ ]:
class TiSASRec(nn.Module):
    """TiSASRec: Time Interval Aware Self-Attentive Sequential Recommendation."""
    
    def __init__(self, n_items, max_len=40, embed_dim=64, n_heads=2, n_layers=2, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.embed_dim = embed_dim
        
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        
        # Time-aware attention layers
        self.attention_layers = nn.ModuleList([
            TimeAwareAttention(embed_dim, n_heads, max_len, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.norm_layers = nn.ModuleList([
            nn.LayerNorm(embed_dim) for _ in range(n_layers)
        ])
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim * 4, embed_dim),
                nn.Dropout(dropout)
            ) for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(embed_dim)
        self._init_weights()
    
    def _init_weights(self):
        nn.init.xavier_uniform_(self.item_emb.weight[1:])
    
    def _get_causal_mask(self, seq_len):
        return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    
    def forward(self, item_seq, time_gaps):
        """
        item_seq: (batch, max_len) 1-indexed item IDs
        time_gaps: (batch, max_len, max_len) pairwise time gaps
        """
        x = self.item_emb(item_seq)
        x = self.emb_dropout(x)
        
        seq_len = item_seq.size(1)
        causal_mask = self._get_causal_mask(seq_len).to(item_seq.device)
        
        for attn, norm, ffn in zip(self.attention_layers, self.norm_layers, self.ffn_layers):
            residual = x
            x = norm(x)
            x = residual + attn(x, time_gaps, causal_mask)
            x = x + ffn(x)
        
        return self.final_norm(x)
    
    def predict(self, item_seq, time_gaps):
        hidden = self.forward(item_seq, time_gaps)
        last_hidden = hidden[:, -1, :]
        item_embs = self.item_emb.weight[1:self.n_items + 1]
        return last_hidden @ item_embs.T

# Quick test
tisasrec = TiSASRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN)
test_seq = torch.tensor([[0, 0, 1, 5, 10], [3, 7, 2, 8, 15]])
test_gaps = torch.rand(2, 5, 5) * 3.0
logits = tisasrec.predict(test_seq, test_gaps)
print(f"TiSASRec output shape: {logits.shape}")
print(f"TiSASRec params: {sum(p.numel() for p in tisasrec.parameters()):,}")

## 5. Dataset with Temporal Information

In [ ]:
class TemporalSeqDataset(Dataset):
    """Dataset that includes timestamps for time-aware models."""
    
    def __init__(self, data, max_len=40):
        self.data = data
        self.max_len = max_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        items = self.data[idx]["items"]
        timestamps = self.data[idx]["timestamps"]
        
        inp_items = items[:-1][-self.max_len:]
        inp_times = timestamps[:-1][-self.max_len:]
        target = items[-1]
        length = len(inp_items)
        
        # Left-pad items (1-indexed)
        pad_len = self.max_len - length
        padded_items = [0] * pad_len + [x + 1 for x in inp_items]
        padded_times = [0.0] * pad_len + inp_times
        
        # Compute pairwise time gaps
        times_tensor = torch.tensor(padded_times, dtype=torch.float)
        time_gaps = torch.abs(times_tensor.unsqueeze(0) - times_tensor.unsqueeze(1))
        
        return (
            torch.tensor(padded_items, dtype=torch.long),
            time_gaps,
            torch.tensor(target, dtype=torch.long),
        )

train_data = temporal_data[:640]
val_data = temporal_data[640:720]
test_data = temporal_data[720:]

train_ds = TemporalSeqDataset(train_data, max_len=MAX_SEQ_LEN)
val_ds = TemporalSeqDataset(val_data, max_len=MAX_SEQ_LEN)
test_ds = TemporalSeqDataset(test_data, max_len=MAX_SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

## 6. Training and Evaluation

In [ ]:
def train_temporal_model(model, train_loader, val_loader, n_epochs=12, lr=0.001):
    """Train a temporal model (TiSASRec)."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98))
    criterion = nn.CrossEntropyLoss()
    
    history = {"loss": [], "val_hr10": []}
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        n_batches = 0
        
        for item_seq, time_gaps, target in train_loader:
            logits = model.predict(item_seq, time_gaps)
            loss = criterion(logits, target)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        history["loss"].append(avg_loss)
        
        # Validation
        hr10 = eval_temporal(model, val_loader, k=10)
        history["val_hr10"].append(hr10)
        
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} — Loss: {avg_loss:.4f}, Val HR@10: {hr10:.4f}")
    
    return history

@torch.no_grad()
def eval_temporal(model, loader, k=10):
    model.eval()
    hits = 0
    total = 0
    for item_seq, time_gaps, target in loader:
        logits = model.predict(item_seq, time_gaps)
        _, topk = logits.topk(k, dim=-1)
        for i in range(target.size(0)):
            if target[i].item() in topk[i].tolist():
                hits += 1
            total += 1
    return hits / total

torch.manual_seed(SEED)
tisasrec_model = TiSASRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN, embed_dim=64, n_heads=2, n_layers=2)
history = train_temporal_model(tisasrec_model, train_loader, val_loader, n_epochs=12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(history["loss"]) + 1)
axes[0].plot(epochs, history["loss"], marker="o", color="steelblue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("TiSASRec Training Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["val_hr10"], marker="s", color="coral")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("HR@10")
axes[1].set_title("TiSASRec Validation HR@10")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Dynamic User Modeling

Instead of a static user embedding, we can model the user representation as **evolving over time**:

$$\mathbf{u}_t = f(\mathbf{u}_{t-1}, \mathbf{e}_{i_t}, \Delta t)$$

This can be implemented with a time-gated GRU:

$$\mathbf{z}_t = \sigma(\mathbf{W}_z [\mathbf{e}_{i_t}; \mathbf{u}_{t-1}; \delta_t])$$
$$\mathbf{u}_t = (1 - \mathbf{z}_t) \odot \mathbf{u}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{u}}_t$$

where $\delta_t$ is a time decay factor.

> **⚠️ Common Pitfall:** Naive time decay (e.g., exponential decay of old interactions) can be too aggressive, forgetting stable long-term preferences. A learned gating mechanism lets the model decide what to retain.

In [ ]:
class TimeGatedGRU(nn.Module):
    """GRU with time-aware gating for dynamic user modeling."""
    
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Standard GRU gates + time input
        gate_input_dim = input_dim + hidden_dim + 1  # +1 for time delta
        self.W_z = nn.Linear(gate_input_dim, hidden_dim)  # Update gate
        self.W_r = nn.Linear(gate_input_dim, hidden_dim)  # Reset gate
        self.W_h = nn.Linear(input_dim + hidden_dim + 1, hidden_dim)  # Candidate
    
    def forward(self, x_seq, time_deltas, h_init=None):
        """
        x_seq: (batch, seq_len, input_dim)
        time_deltas: (batch, seq_len) time gap from previous step
        Returns: (batch, seq_len, hidden_dim) hidden states at each step
        """
        batch_size, seq_len, _ = x_seq.shape
        
        if h_init is None:
            h = torch.zeros(batch_size, self.hidden_dim, device=x_seq.device)
        else:
            h = h_init
        
        outputs = []
        for t in range(seq_len):
            x_t = x_seq[:, t, :]  # (batch, input_dim)
            dt = time_deltas[:, t:t+1]  # (batch, 1)
            
            gate_input = torch.cat([x_t, h, dt], dim=-1)
            z = torch.sigmoid(self.W_z(gate_input))  # Update gate
            r = torch.sigmoid(self.W_r(gate_input))  # Reset gate
            
            candidate_input = torch.cat([x_t, r * h, dt], dim=-1)
            h_tilde = torch.tanh(self.W_h(candidate_input))
            
            h = (1 - z) * h + z * h_tilde
            outputs.append(h.unsqueeze(1))
        
        return torch.cat(outputs, dim=1)

# Test
tgru = TimeGatedGRU(input_dim=64, hidden_dim=128)
test_x = torch.randn(4, 10, 64)
test_dt = torch.rand(4, 10)  # Time deltas
out = tgru(test_x, test_dt)
print(f"TimeGatedGRU output shape: {out.shape}")

## 8. Temporal Point Processes

**Temporal point processes** model both *what* happens and *when* it happens. The key quantity is the **conditional intensity function**:

$$\lambda^*(t) = \lambda_0 + \sum_{t_i < t} \alpha \cdot \exp(-\beta(t - t_i))$$

This is the **Hawkes process**: past events excite future event rates, with exponential decay.

> **🔑 Pro Tip:** In recommendation, a Hawkes-style model can predict not just what a user will interact with next, but also *when* they will return. This is valuable for notification timing and session prediction.

In [ ]:
class HawkesIntensity(nn.Module):
    """Learnable Hawkes process intensity for temporal modeling."""
    
    def __init__(self, n_event_types=20):
        super().__init__()
        self.base_rate = nn.Parameter(torch.ones(n_event_types) * 0.1)
        self.alpha = nn.Parameter(torch.ones(n_event_types, n_event_types) * 0.5)
        self.beta = nn.Parameter(torch.ones(n_event_types, n_event_types) * 1.0)
    
    def intensity(self, t, history_times, history_types):
        """
        Compute intensity at time t given event history.
        t: scalar, current time
        history_times: (n_events,) past event times
        history_types: (n_events,) past event type IDs
        Returns: (n_event_types,) intensity for each type
        """
        base = F.softplus(self.base_rate)  # Ensure positive
        
        if len(history_times) == 0:
            return base
        
        dt = t - history_times  # (n_events,)
        alpha_pos = F.softplus(self.alpha)  # (n_types, n_types)
        beta_pos = F.softplus(self.beta)
        
        # Sum excitation from all past events
        excitation = torch.zeros_like(base)
        for i, (ti, ki) in enumerate(zip(history_times, history_types)):
            decay = torch.exp(-beta_pos[:, ki] * (t - ti))
            excitation += alpha_pos[:, ki] * decay
        
        return base + excitation

# Demo: visualize Hawkes intensity
hawkes = HawkesIntensity(n_event_types=5)
event_times = torch.tensor([0.0, 0.5, 0.8, 2.0, 2.1, 5.0])
event_types = torch.tensor([0, 0, 1, 2, 0, 1])

t_range = np.linspace(0, 7, 200)
intensities = []
for t in t_range:
    mask = event_times < t
    with torch.no_grad():
        lam = hawkes.intensity(
            torch.tensor(t),
            event_times[mask],
            event_types[mask]
        )
    intensities.append(lam[0].item())  # Type 0 intensity

plt.figure(figsize=(10, 4))
plt.plot(t_range, intensities, color="steelblue", label="Type 0 intensity")
for t in event_times:
    plt.axvline(t.item(), color="gray", linestyle="--", alpha=0.4)
type0_events = event_times[event_types == 0]
plt.scatter(type0_events.numpy(), [0.05] * len(type0_events), color="red", s=50, zorder=5, label="Type 0 events")
plt.xlabel("Time")
plt.ylabel("Intensity")
plt.title("Hawkes Process: Event Intensity Over Time")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Periodic Patterns

Users often exhibit periodic behavior (daily, weekly, seasonal). We can encode these patterns.

> **💡 Concept:** Adding periodic features (hour of day, day of week) as input features is a simple but effective way to capture temporal patterns without complex architectures.

In [ ]:
def encode_periodic_features(timestamps_days, embed_dim=16):
    """Encode periodic features from timestamps.
    
    timestamps_days: (batch, seq_len) float timestamps in days
    Returns: (batch, seq_len, embed_dim) periodic encoding
    """
    # Extract periodic components
    hour_of_day = (timestamps_days % 1) * 24  # 0-24
    day_of_week = timestamps_days % 7  # 0-7
    
    features = []
    n_feats = embed_dim // 4  # 4 features per component
    
    for period, signal in [(24.0, hour_of_day), (7.0, day_of_week)]:
        for i in range(n_feats):
            freq = 2 * math.pi * (i + 1) / period
            features.append(torch.sin(signal * freq))
            features.append(torch.cos(signal * freq))
    
    return torch.stack(features, dim=-1)  # (batch, seq_len, embed_dim)

# Visualize periodic patterns
t = torch.linspace(0, 14, 500).unsqueeze(0)  # 14 days
periodic = encode_periodic_features(t, embed_dim=8)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for i in range(4):
    axes[0].plot(t[0].numpy(), periodic[0, :, i].numpy(), alpha=0.7, label=f"Feature {i}")
axes[0].set_xlabel("Time (days)")
axes[0].set_title("Daily Periodic Features")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i in range(4, 8):
    axes[1].plot(t[0].numpy(), periodic[0, :, i].numpy(), alpha=0.7, label=f"Feature {i}")
axes[1].set_xlabel("Time (days)")
axes[1].set_title("Weekly Periodic Features")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Exercises

### 🏋️ Exercise 1: Add Time-Aware Attention to SASRec (TiSASRec-style)

Modify a standard SASRec to accept timestamps and add time-interval bias to attention.

In [ ]:
# 🏋️ Exercise 1: Simplified TiSASRec
# 
# Modify the SASRec from Chapter 5.2 to incorporate time intervals.
# Approach: Add a time-interval bias term to the attention scores.

class SimpleTiSASRec(nn.Module):
    def __init__(self, n_items, max_len=40, embed_dim=64, n_heads=2, n_layers=2,
                 n_time_buckets=32, dropout=0.2):
        super().__init__()
        # TODO: Implement
        # 1. Standard SASRec components (item_emb, pos_emb, transformer layers)
        # 2. Time-interval embedding: n_time_buckets -> 1 (scalar bias per bucket)
        # 3. In forward(), compute time bucket IDs from pairwise time gaps
        # 4. Add time bias to attention scores (before softmax)
        #
        # Hint: You'll need to use hooks or custom attention rather than
        # nn.TransformerEncoder, since you need to modify the attention scores.
        pass
    
    def predict(self, item_seq, time_gaps):
        # TODO: Implement
        pass

# Test:
# model = SimpleTiSASRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN)
# logits = model.predict(test_seq, test_gaps)
# print(f"Output: {logits.shape}")

### 🏋️ Exercise 2: Compare SASRec vs TiSASRec

Train both models on the temporal dataset and compare their performance.

In [ ]:
# 🏋️ Exercise 2: SASRec vs TiSASRec Comparison

# TODO:
# 1. Adapt the SASRec model from Chapter 5.2 to work with this dataset
#    (it should ignore time_gaps, only use item_seq)
# 2. Train both models for the same number of epochs
# 3. Plot validation HR@10 curves for both models
# 4. Report final test metrics
#
# Expected result: TiSASRec should perform slightly better, especially
# for cases where time gaps carry important signal.
pass

### 🏋️ Exercise 3: Implement Time-Decay Attention

A simpler alternative to TiSASRec: multiply attention weights by an exponential time decay.

In [ ]:
# 🏋️ Exercise 3: Time-Decay Attention

class TimeDecayAttention(nn.Module):
    """Self-attention with exponential time decay.
    
    Attention is multiplied by exp(-gamma * delta_t) where gamma is learnable.
    """
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        # TODO: Implement
        # 1. Standard Q, K, V projections
        # 2. Learnable decay rate gamma (one per head, initialized to 1.0)
        # 3. In forward: 
        #    attn_scores = Q @ K^T / sqrt(d_k)
        #    time_decay = exp(-softplus(gamma) * time_gaps)
        #    attn_scores = attn_scores * time_decay
        #    Apply causal mask, softmax, weighted sum
        pass
    
    def forward(self, x, time_gaps, causal_mask=None):
        # TODO: Implement
        pass

# Test:
# decay_attn = TimeDecayAttention(embed_dim=64, n_heads=2)
# out = decay_attn(test_x, test_gaps[:, :10, :10])
# print(f"Output shape: {out.shape}")

## Summary

| Method | Year | Time Encoding | Key Idea |
|--------|------|--------------|----------|
| SASRec | 2018 | Positional only | Causal self-attention |
| TiSASRec | 2020 | Relative time buckets | Time-interval bias in attention |
| Time-Gated GRU | Various | Continuous time input | Adaptive gating with time |
| Hawkes + Rec | Various | Intensity function | Temporal point process |

**Key Takeaways:**
1. Time intervals between interactions carry important information about user intent
2. TiSASRec adds time-interval embeddings to both keys and values in attention
3. Log-scale bucketing of time gaps is effective for handling the wide range of intervals
4. Periodic features (hour, day-of-week) capture routine behavior patterns
5. Temporal point processes jointly model what and when, useful for proactive recommendation
6. Next chapter: **Contrastive learning** to address data sparsity in sequential rec